# Zastosowanie uczenia maszynowego do wykrywania anomalii w logach audytu systemu Linux

Autorzy:
- Lukasz Kierzek
- Tomasz Gondek

Cel projektu: porownanie prostego podejscia regułowego z metodami ML: Isolation Forest oraz Local Outlier Factor.

## 1. Wczytanie danych i bibliotek

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from config import RAW_LOG_PATH
from src.audit_parser import parse_line
from src.features import records_to_features
from src.log_loader import load_logs
from src.ml_detector import (
    predict_anomalies,
    predict_local_outlier_factor,
    train_isolation_forest,
    train_local_outlier_factor,
)
from src.preprocessing import (
    calculate_baseline_metrics,
    calculate_detection_statistics,
    compare_detection_methods,
    detections_to_dataframe,
    get_rule_anomaly_indices,
)
from src.reporting import print_detection_results
from src.rules import run_all_rules_with_context

logs = load_logs(RAW_LOG_PATH)
records = [parse_line(line) for line in logs if line.strip()]

print(f"Loaded logs: {len(records)}")

## 2. Przykladowe rekordy po parsowaniu

In [ ]:
for record in records[:5]:
    print(record)

## 3. Detekcja regulowa

In [ ]:
detections_by_record = run_all_rules_with_context(records)

for record, detections in zip(records, detections_by_record):
    if detections:
        print_detection_results(record, detections)

## 4. Statystyki regul

In [ ]:
stats = calculate_detection_statistics(records)
stats

In [ ]:
summary = pd.DataFrame([
    {"metric": "total_logs", "value": stats["total_logs"]},
    {"metric": "anomaly_logs", "value": stats["anomaly_logs"]},
    {"metric": "normal_logs", "value": stats["normal_logs"]},
    {
        "metric": "anomaly_percentage",
        "value": round(stats["anomaly_logs"] / stats["total_logs"] * 100, 2) if stats["total_logs"] else 0,
    },
])
summary

## 5. Tabela wykrytych anomalii

In [ ]:
detections_df = detections_to_dataframe(records)
detections_df.head(15)

## 6. Liczba wykryc dla poszczegolnych regul

In [ ]:
rule_summary = (
    detections_df["rule"]
    .value_counts()
    .reset_index()
)
rule_summary.columns = ["rule", "detections"]
rule_summary

In [ ]:
if not rule_summary.empty:
    rule_summary.plot.bar(x="rule", y="detections", legend=False)
    plt.title("Detected anomalies by rule")
    plt.xlabel("Rule")
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Rozklad aktywnosci w czasie

In [ ]:
hours = [record.timestamp.hour for record in records if record.timestamp]

plt.hist(hours, bins=24)
plt.title("Activity by hour")
plt.xlabel("Hour")
plt.ylabel("Number of logs")
plt.show()

## 8. Feature engineering

In [ ]:
features_df = records_to_features(records)
features_df.head()

In [ ]:
features_df.describe().T

## 9. Isolation Forest

In [ ]:
isolation_model = train_isolation_forest(features_df)
isolation_predictions = predict_anomalies(isolation_model, features_df)

isolation_anomalies = features_df.copy()
isolation_anomalies["prediction"] = isolation_predictions
isolation_anomalies[isolation_anomalies["prediction"] == -1]

## 10. Local Outlier Factor

In [ ]:
lof_model = train_local_outlier_factor(features_df)
lof_predictions = predict_local_outlier_factor(lof_model, features_df)

lof_anomalies = features_df.copy()
lof_anomalies["prediction"] = lof_predictions
lof_anomalies[lof_anomalies["prediction"] == -1]

## 11. Porownanie metod

In [ ]:
comparison_df = compare_detection_methods(
    records,
    isolation_predictions,
    lof_predictions,
)
comparison_df

In [ ]:
plt.bar(
    comparison_df["method"],
    comparison_df["detected_anomalies"],
)
plt.title("Comparison of anomaly detection methods")
plt.ylabel("Detected anomalies")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 12. Czesc wspolna i roznice

In [ ]:
rule_indices = get_rule_anomaly_indices(records)
isolation_indices = {index for index, value in enumerate(isolation_predictions) if value == -1}
lof_indices = {index for index, value in enumerate(lof_predictions) if value == -1}

venn_like_df = pd.DataFrame([
    {"category": "Rule only", "count": len(rule_indices - isolation_indices - lof_indices)},
    {"category": "Isolation only", "count": len(isolation_indices - rule_indices - lof_indices)},
    {"category": "LOF only", "count": len(lof_indices - rule_indices - isolation_indices)},
    {"category": "All methods", "count": len(rule_indices & isolation_indices & lof_indices)},
])
venn_like_df

## 13. Automatyczne podsumowanie wynikow

In [ ]:
best_by_count = comparison_df.sort_values("detected_anomalies", ascending=False).iloc[0]

print("Podsumowanie projektu")
print(f"Liczba analizowanych logow: {len(records)}")
print(f"Reguly wykryly: {stats['anomaly_logs']} anomalii")
print(
    "Isolation Forest wykryl: "
    f"{int((isolation_predictions == -1).sum())} anomalii"
)
print(
    "Local Outlier Factor wykryl: "
    f"{int((lof_predictions == -1).sum())} anomalii"
)
print(
    "Najwiecej zdarzen oznaczyla metoda: "
    f"{best_by_count['method']} ({best_by_count['detected_anomalies']})"
)
print("Wniosek: reguly sa czytelne i dobrze wykrywaja znane wzorce, a ML wskazuje zdarzenia nietypowe wzgledem cech liczbowych.")

## 14. Metryki prostej reguly bazowej

Metryki ponizej sa liczone wzgledem listy podejrzanych komend. To uproszczona ocena, bo projekt nie posiada pelnych etykiet ground truth dla kazdej linii logu.

In [ ]:
baseline_metrics = calculate_baseline_metrics(records)
baseline_metrics